# Tool 1: Vector Search

In [0]:
spark.sql(r"""
CREATE OR REPLACE FUNCTION workspace.default.vector_search(
    question      STRING COMMENT 'The medical or facility question to search semantically.
                                  Examples:
                                  "hospitals with ICU beds"
                                  "dialysis treatment facilities"
                                  "clinics in Dansoman with heart treatment"
                                  "emergency care near Kejetia"
                                  Do NOT use for counting or aggregation — use sql_query instead.',

    location      STRING COMMENT 'Optional. Any location string to filter by.
                                  Matched as substring against osm_display_name and
                                  resolved_location_query — so any level of specificity works.
                                  Pass NULL to search all locations.
                                  Examples:
                                  "Dansoman"              — neighbourhood
                                  "Accra"                 — city
                                  "Kumasi"                — city
                                  "Greater Accra Region"  — region
                                  "Ablekuma"              — district
                                  "Ashanti"               — partial region name
                                  "Tamale"                — city
                                  Unlike region/city filters, this matches ANY location
                                  at ANY administrative level.',

    facility_type STRING COMMENT 'Optional. Filter by facility type.
                                  Pass NULL for all types.
                                  Valid values: hospital, clinic, pharmacy, doctor, dentist.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Retrieves semantically relevant Ghana medical facility documents.
         Returns RAW facility data — no LLM generation.
         Agent uses results to reason and synthesize its own answer.

         LOCATION FILTER works as substring match against osm_display_name
         and resolved_location_query — so it handles neighbourhoods, districts,
         cities, and regions equally well.

         facility_type use exact match on index metadata.

         FILTER EXTRACTION RULES:
         "clinics in Dansoman"            → location="Dansoman", facility_type="clinic"
         "hospitals in Accra"      → location="Accra", facility_type="hospital"
         "facilities near Kejetia Market" → location="Kejetia"
         "hospitals in Ashanti"           → location="Ashanti", facility_type="hospital"
         "clinics in Kumasi"      → location="Kumasi", facility_type="clinic",
         "dialysis anywhere in Ghana"     → location=NULL

         USE FOR: capability/procedure/equipment/specialty questions
         DO NOT USE FOR: counting (sql_query), single facility (get_facility)

         RETURNS: JSON with
         - facilities: top 5 most relevant after location post-filtering
         - num_results: count returned
         - num_before_location_filter: count before location filter (debug)
         - filters_applied: which filters were active
         - query_used: question used for vector search'
AS $$
import json
import re
import requests

WORKSPACE_URL = "your_WORKSPACE_URL"
VECTOR_SEARCH_ENDPOINT  = "your_VECTOR_SEARCH_ENDPOINT"
INDEX_NAME    = "your_INDEX_NAME"
DATABRICKS_TOKEN = "your_DATABRICKS_TOKEN"

def clean_coord(val):
    if val is None:
        return None
    s = str(val).strip()
    if s in ("", "0", "0.0", "None", "null"):
        return None
    try:
        f = float(s)
        return f if f != 0.0 else None
    except (ValueError, TypeError):
        return None

try:
    # ── Build exact-match filters ─────────────────────────────
    filters_applied = {}
    filters_json    = {}

    if facility_type:
        ft = facility_type.lower().strip()
        filters_json["facility_type"] = ft
        filters_applied["facility_type"] = ft

    if location:
        filters_applied["location"] = location

    # ── Build REST payload ────────────────────────────────────
    # Vector Search REST API endpoint
    url = (
        f"https://{WORKSPACE_URL}/api/2.0/vector-search/indexes"
        f"/{INDEX_NAME}/query"
    )

    num_to_retrieve = 15 if location else 5

    payload = {
        "query_text"  : question,
        "columns"     : [
            "id", "name", "document",
            "facility_type", "organization_type",
            "address_city", "address_stateOrRegion", "address_country",
            "osm_display_name", "resolved_location_query",
            "numberDoctors", "capacity", "lat", "lon"
        ],
        "num_results" : num_to_retrieve
    }

    # Add filters if any exact-match filters exist
    if filters_json:
        # Vector Search REST API expects filters as AND conditions
        # Format: {"field": "value"} for each filter
        payload["filters_json"] = json.dumps(filters_json)

    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type" : "application/json"
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=30)

    if resp.status_code != 200:
        return json.dumps({
            "error"       : f"Vector search failed: {resp.status_code} — {resp.text}",
            "facilities"  : [],
            "num_results" : 0
        })

    data = resp.json()

    # ── Parse results ─────────────────────────────────────────
    # REST API returns same structure as SDK
    all_facilities = []

    result  = data.get("result", {})
    rows    = result.get("data_array", [])
    columns = [
        col["name"]
        for col in result.get("manifest", {}).get("columns", [])
    ]

    for i, row in enumerate(rows):
        if columns:
            doc = dict(zip(columns, row))
        else:
            # Positional fallback
            doc = {
                "id"                     : row[0],
                "name"                   : row[1],
                "document"               : row[2],
                "facility_type"          : row[3],
                "organization_type"      : row[5],
                "address_city"           : row[6],
                "address_stateOrRegion"  : row[7],
                "address_country"        : row[8],
                "osm_display_name"       : row[9],
                "resolved_location_query": row[10],
                "numberDoctors"          : row[11],
                "capacity"               : row[12],
                "lat"                    : row[13],
                "lon"                    : row[14],
                "score"                  : row[15] if len(row) > 15 else None
            }

        lat_clean = clean_coord(doc.get("lat"))
        lon_clean = clean_coord(doc.get("lon"))

        all_facilities.append({
            "rank"                   : i + 1,
            "id"                     : str(doc.get("id", "")),
            "name"                   : doc.get("name", ""),
            "facility_type"          : doc.get("facility_type") or "",
            "city"                   : doc.get("address_city", ""),
            "region"                 : doc.get("address_stateOrRegion", ""),
            "country"                : doc.get("address_country", "Ghana"),
            "osm_display_name"       : doc.get("osm_display_name", ""),
            "resolved_location_query": doc.get("resolved_location_query", ""),
            "number_doctors"         : doc.get("numberDoctors", 0),
            "capacity"               : doc.get("capacity", 0),
            "lat"                    : lat_clean,
            "lon"                    : lon_clean,
            "has_location"           : lat_clean is not None and lon_clean is not None,
            "relevance_score"        : round(float(doc.get("score", 0) or 0), 4),
            "document"               : str(doc.get("document", ""))[:800]
        })

    num_before_filter = len(all_facilities)

    # ── Location post-filter — substring match ────────────────
    if location:
        loc_lower      = location.lower().strip()
        loc_normalized = re.sub(r'[-_]', ' ', loc_lower)
        loc_normalized = re.sub(r'\s+',  ' ', loc_normalized).strip()

        def location_matches(f):
            fields_to_check = [
                f.get("osm_display_name",        ""),
                f.get("resolved_location_query",  ""),
                f.get("city",                     ""),
                f.get("region",                   ""),
            ]
            return any(
                loc_normalized in re.sub(r'\s+', ' ', str(field).lower().strip())
                for field in fields_to_check
                if field
            )

        filtered = [f for f in all_facilities if location_matches(f)]
        for i, f in enumerate(filtered):
            f["rank"] = i + 1
        facilities = filtered[:5]
    else:
        facilities = all_facilities[:5]

    return json.dumps({
        "facilities"                : facilities,
        "num_results"               : len(facilities),
        "num_before_location_filter": num_before_filter,
        "filters_applied"           : filters_applied,
        "query_used"                : question
    })

except Exception as e:
    return json.dumps({
        "error"                     : str(e),
        "facilities"                : [],
        "num_results"               : 0,
        "num_before_location_filter": 0,
        "filters_applied"           : {}
    })
$$
""")

print("vector_search UC Function created successfully")

# Tool 2: Genie Text2SQL

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.sql_query(
    question STRING COMMENT 'Plain English question that requires structured data analysis.
                             Examples:
                             "how many hospitals are in Greater Accra Region",
                             "How many hospitals in Accra have the ability to perform treatments related to the Heart ?",
                             "How many hospitals have cardiology?",
                             "list all public hospitals in Ashanti Region",
                             "which region has the most healthcare facilities",
                             "find regions with fewer than 3 hospitals",
                             Be specific — include filters, groupings, or conditions 
                             you want in the SQL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Converts plain English to SQL and runs it against the Ghana facilities Delta Table.
         
         USE THIS TOOL WHEN the question involves:
         - Counting: how many, total number of
         - Filtering: list all X in Y, facilities where Z
         - Aggregating: average, sum, maximum, minimum
         - Ranking: which region has most/least, top N
         - Comparing numbers across regions or facility types
         - Any question that needs exact numbers from structured columns
                  
         RETURNS: JSON string with fields:
         - answer: natural language description of the result
         - sql_query: the SQL that was executed
         - results: list of result rows as dicts (max 20 rows)
         - num_rows: total number of rows returned'
AS $$
import requests
import json
import time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

try:
    # STEP 1: Start conversation
    start = requests.post(
        f"{BASE}/start-conversation",
        headers=headers,
        json={"content": question},
        timeout=30
    )

    if start.status_code != 200:
        return json.dumps({"error": f"Genie failed: {start.text}"})

    data = start.json()
    conv_id = data["conversation_id"]
    msg_id = data["message_id"]

    # STEP 2: Poll
    message = None
    for _ in range(30):
        poll = requests.get(
            f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
            headers=headers,
            timeout=30
        )

        message = poll.json()
        status = message.get("status", "")

        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie {status}", "raw": message})

        time.sleep(2)

    # STEP 3: Parse attachments
    attachments = message.get("attachments", [])

    sql_query = None
    description = None
    text_answer = ""

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")
            description = a["query"].get("description")

        if "text" in a:
            text_answer += a["text"].get("content", "") + " "

    text_answer = text_answer.strip()

    # If no SQL, return text
    if not sql_query:
        return json.dumps({
            "answer": text_answer or "No answer returned",
            "sql_query": None,
            "results": [],
            "num_rows": 0
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    return json.dumps({
        "answer": text_answer or description or f"{len(results)} rows returned",
        "sql_query": sql_query,
        "results": results[:20],
        "num_rows": len(results)
    })

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")

print("sql_query function created")

# Tool 3 Get Facility Details

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_facility(
    name STRING COMMENT 'Name of the specific facility to look up.
                         Can be exact or partial name.
                         Examples: "2BN Military Hospital", "Abuakwa Maternity Home".
                         Use when you already know the facility name from 
                         a previous rag_search or sql_query result.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Fetch complete structured details for one specific Ghana medical facility by name.
         
         USE THIS TOOL WHEN:
         - You found a facility name from rag_search and need its exact structured data
         - You need to verify specific claims about a named facility
         - You need the exact lat/lon of a specific facility for geospatial calculations
         - You need to cross-reference a facility found in one tool with structured data
         - A user asks specifically about one named facility
         
         DO NOT USE THIS TOOL FOR:
         - Searching across multiple facilities (use rag_search instead)
         - Counting or aggregating (use sql_query instead)
         
         RETURNS: JSON string with fields:
         - found: true/false
         - facility: dict with all columns for this facility
         - lat: latitude as float (0.0 if unknown)
         - lon: longitude as float (0.0 if unknown)'
AS $$
import json

import requests, time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

question = (
    f"Get all details for the facility named '{name}'. "
    f"Return name, facilityTypeId, operatorTypeId, address_city, "
    f"address_stateOrRegion, numberDoctors, capacity, lat, lon, "
    f"specialties, capability, description."
)

try:
    start   = requests.post(f"{BASE}/start-conversation",
                            headers=headers, json={"content": question}, timeout=30)
    
    if start.status_code != 200:
        return json.dumps({"error": f"Failed to start: {start.text}"})
        
    conv_id = start.json()["conversation_id"]
    msg_id  = start.json()["message_id"]
    message = None

    for _ in range(30):
        poll   = requests.get(f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
                              headers=headers, timeout=30)
        message = poll.json()
        status = message.get("status")
        
        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie query failed: {status}"})
            
        time.sleep(2)
    
    attachments = message.get("attachments", [])
    sql_query = None

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")

    # FIX #2: Return correct schema if no SQL
    if not sql_query:
        return json.dumps({
            "found": False,
            "facility": {},
            "error": "No SQL generated"
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    # FIX #1: Correct Indentation
    if results:
        f = results[0]
        
        # FIX #3: Safely convert to float
        try:
            lat = float(f.get("lat")) if f.get("lat") else 0.0
        except (ValueError, TypeError):
            lat = 0.0
            
        try:
            lon = float(f.get("lon")) if f.get("lon") else 0.0
        except (ValueError, TypeError):
            lon = 0.0

        return json.dumps({
            "found"   : True,
            "facility": f,
            "lat"     : lat,
            "lon"     : lon
        })
        
    return json.dumps({"found": False, "facility": {}})

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")
print("created get_facility function")


# Tool 4: External Data


In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.external_data(
    data_type STRING COMMENT 'The type of reference data to retrieve. Must be one of:
                              "population"      - Ghana region population figures (2021 census)
                              "area"            - Region area in square kilometres
                              "capital"         - Regional capital city name and GPS coordinates
                              "who_standards"   - WHO minimum healthcare benchmarks
                              "health_context"  - Ghana health system context and averages
                              "derived_metrics" - Pre-calculated: hospitals needed, doctors needed
                                                  beds needed per region based on WHO standards
                              "all_regions"     - Population + area + capital for all 16 regions',

    region   STRING COMMENT 'Optional. Specific Ghana region to get data for.
                             Pass NULL to get data for all regions.
                             Example: "Northern Region", "Upper East Region".
                             Only applicable for data_type: population, area, capital, 
                             derived_metrics.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns static reference data about Ghana regions and WHO healthcare standards.
         No API call — instant response from pre-loaded reference data.
         
         USE THIS TOOL WHEN:
         - You need Ghana region populations for per-capita calculations
         - Comparing facility counts against WHO minimum standards
         - You need coordinates of a regional capital for geospatial queries
         - Calculating healthcare deficits (hospitals needed vs actual)
         - Any question involving "per capita", "per 100,000", "relative to population"
         - Identifying medical deserts using WHO benchmarks
         
         ALWAYS CALL THIS BEFORE doing per-capita calculations —
         do not estimate or guess population figures.
         
         COMMON PATTERN:
         1. Call sql_query to get facility counts per region
         2. Call external_data("derived_metrics") to get hospitals_needed_who per region
         3. Compare actual vs needed to identify underserved regions
         
         WHO STANDARDS included:
         - Minimum 1 hospital per 100,000 people
         - Minimum 1 physician per 1,000 people  
         - Minimum 1 hospital bed per 1,000 people
         
         RETURNS: JSON string — structure depends on data_type requested.'
AS $$
import json

GHANA_DATA = {
    "populations": {
        "Greater Accra Region": 5455692, "Ashanti Region": 5440463,
        "Eastern Region": 2925653,       "Central Region": 2859821,
        "Northern Region": 2310939,      "Western Region": 2060585,
        "Volta Region": 1652772,         "Upper East Region": 1301226,
        "Bono Region": 1208649,          "Bono East Region": 1202739,
        "Upper West Region": 901502,      "Western North Region": 880921,
        "Oti Region": 747248,            "Savannah Region": 653266,
        "North East Region": 650856,     "Ahafo Region": 564662,
    },
    "area_km2": {
        "Savannah Region": 35862,      "Northern Region": 25448,
        "Ashanti Region": 24389,       "Bono East Region": 23257,
        "Eastern Region": 19323,       "Upper West Region": 18476,
        "Volta Region": 9504,          "Western Region": 13847,
        "North East Region": 9074,     "Upper East Region": 8842,
        "Bono Region": 11107,          "Oti Region": 11066,
        "Western North Region": 10074, "Central Region": 9826,
        "Ahafo Region": 5193,          "Greater Accra Region": 3245,
    },
    "capitals": {
        "Greater Accra Region": {"city": "Accra",        "lat": 5.6037,  "lon": -0.1870},
        "Ashanti Region":       {"city": "Kumasi",       "lat": 6.6885,  "lon": -1.6244},
        "Western Region":       {"city": "Sekondi",      "lat": 4.9341,  "lon": -1.7097},
        "Eastern Region":       {"city": "Koforidua",    "lat": 6.0940,  "lon": -0.2574},
        "Central Region":       {"city": "Cape Coast",   "lat": 5.1054,  "lon": -1.2466},
        "Northern Region":      {"city": "Tamale",       "lat": 9.4008,  "lon": -0.8393},
        "Upper East Region":    {"city": "Bolgatanga",   "lat": 10.7856, "lon": -0.8514},
        "Upper West Region":    {"city": "Wa",           "lat": 10.0601, "lon": -2.5099},
        "Volta Region":         {"city": "Ho",           "lat": 6.6011,  "lon":  0.4703},
        "Bono Region":          {"city": "Sunyani",      "lat": 7.3349,  "lon": -2.3269},
        "Bono East Region":     {"city": "Techiman",     "lat": 7.5833,  "lon": -1.9333},
        "Ahafo Region":         {"city": "Goaso",        "lat": 6.8000,  "lon": -2.5167},
        "Oti Region":           {"city": "Dambai",       "lat": 8.0667,  "lon":  0.1833},
        "Savannah Region":      {"city": "Damongo",      "lat": 9.0833,  "lon": -1.8167},
        "North East Region":    {"city": "Nalerigu",     "lat": 10.5306, "lon": -0.3686},
        "Western North Region": {"city": "Sefwi Wiawso", "lat": 6.1667,  "lon": -2.4833},
    },
    "who_standards": {
        "physicians_per_1000": 1.0,
        "hospital_beds_per_1000": 1.0,
        "hospitals_per_100000": 1.0,
        "nurses_per_1000": 3.0
    }
}

who = GHANA_DATA["who_standards"]

if data_type == "population":
    if region:
        pop = GHANA_DATA["populations"].get(region)
        return json.dumps({"region": region, "population": pop} if pop
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"populations": GHANA_DATA["populations"]})

elif data_type == "area":
    if region:
        area = GHANA_DATA["area_km2"].get(region)
        return json.dumps({"region": region, "area_km2": area} if area
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"areas": GHANA_DATA["area_km2"]})

elif data_type == "capital":
    if region:
        cap = GHANA_DATA["capitals"].get(region)
        return json.dumps({"region": region, **cap} if cap
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"capitals": GHANA_DATA["capitals"]})

elif data_type == "who_standards":
    return json.dumps({"who_standards": who})

elif data_type == "derived_metrics":
    metrics = {}
    for r, pop in GHANA_DATA["populations"].items():
        area = GHANA_DATA["area_km2"].get(r, 1)
        metrics[r] = {
            "population"           : pop,
            "area_km2"             : area,
            "pop_density_per_km2"  : round(pop / area, 1),
            "hospitals_needed_who" : round((pop / 100000) * who["hospitals_per_100000"]),
            "doctors_needed_who"   : round((pop / 1000)   * who["physicians_per_1000"]),
            "beds_needed_who"      : round((pop / 1000)   * who["hospital_beds_per_1000"]),
        }
    if region:
        return json.dumps({"region": region,
                           "metrics": metrics.get(region, {})})
    return json.dumps({"derived_metrics": metrics})

return json.dumps({"error": f"Unknown data_type: {data_type}"})
$$
""")

print("external_data function created")

# Tool 5: Find nearby facilities

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.find_nearby_facilities(
    center_lat   DOUBLE  COMMENT 'Latitude of the center point to search around.
                                  Get this from external_data("capital") for regional
                                  capitals, or from get_facility() for named facilities.',
    center_lon   DOUBLE  COMMENT 'Longitude of the center point to search around.',
    radius_km    DOUBLE  COMMENT 'Search radius in kilometres. 
                                  Typical values: 10 (urban), 50 (regional), 100 (national).
                                  WHO recommends emergency care within 50km.',
    condition    STRING  COMMENT 'Optional. Medical condition, procedure, or capability to filter by.
                                  Pass NULL to find all facilities within radius.
                                  Examples: "dialysis", "surgery", "maternity", "ICU", "emergency"
                                  This does a keyword match against capability, procedure, 
                                  specialties columns.',
    facility_type STRING COMMENT 'Optional. Filter by facility type.
                                  Pass NULL for all types.
                                  Valid values: hospital, clinic, pharmacy, doctor, dentist'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Find all facilities within a radius of a point that match an optional condition.
         Combines geospatial distance calculation with capability filtering in one call.

         USE THIS TOOL WHEN:
         - "hospitals within X km of Y"
         - "facilities treating Z near location"
         - "what healthcare is accessible from this point"
         - "is there a hospital within 50km of this area"

         WORKFLOW:
         1. Get center coordinates from external_data or get_facility first
         2. Call this tool with those coordinates
         3. Results are already filtered by radius and sorted by distance

         RETURNS: JSON with
         - facilities: list sorted by distance_km (closest first)
                       each has name, city, region, facility_type, 
                       lat, lon, distance_km, matched_capabilities
         - count: total facilities found within radius
         - radius_km: the search radius used
         - nearest: the single closest matching facility'
AS $$
import requests
import json
import math

# WORKSPACE_URL = "your-WORKSPACE_URL"
# WAREHOUSE_ID = "your-WAREHOUSE_ID"
# DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

def haversine(lat1, lon1, lat2, lon2):
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat/2)**2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon/2)**2)
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

try:
    BASE_URL = f"https://{WORKSPACE_URL}/api/2.0/sql/statements"

    # 1. Build the exact SQL Query    
    condition_clause = ""
    if condition:
        c = condition.lower()
        condition_clause = (
            f"AND ("
            f"  exists(capability, x -> x ilike '%{c}%') OR "
            f"  exists(procedure, x -> x ilike '%{c}%') OR "
            f"  exists(specialties, x -> x ilike '%{c}%')"
            f")"
        )

    type_clause = ""
    if facility_type:
        type_clause = f"AND facilityTypeId = '{facility_type}'"

    sql_query = (
        f"SELECT name, facilityTypeId, operatorTypeId, address_city, "
        f"address_stateOrRegion, lat, lon, capability, procedure, specialties "
        f"FROM facilities3 "
        f"WHERE lat IS NOT NULL AND lon IS NOT NULL "
        f"AND lat != 0 AND lon != 0 "
        f"{condition_clause} {type_clause}"
    )

    # 2. Execute directly on the SQL Warehouse
    payload = {
        "warehouse_id": WAREHOUSE_ID,
        "statement": sql_query,
        "wait_timeout": "30s" # Wait synchronously for up to 30 seconds
    }

    start = requests.post(BASE_URL, headers=headers, json=payload, timeout=40)
    
    if start.status_code != 200:
        return json.dumps({"error": f"SQL Execution failed: {start.text}", "count": 0})

    data = start.json()
    status = data.get("status", {}).get("state")
    statement_id = data.get("statement_id")

    # 3. Poll if it didn't finish within the 30s wait_timeout (rare, but safe)
    while status in ("PENDING", "RUNNING"):
        time.sleep(1)
        poll = requests.get(f"{BASE_URL}/{statement_id}", headers=headers, timeout=10)
        data = poll.json()
        status = data.get("status", {}).get("state")

    if status != "SUCCEEDED":
        return json.dumps({"error": f"Query failed with status: {status}", "count": 0})

    # 4. Parse Results
    # The SQL API returns a much simpler JSON format than Genie
    manifest = data.get("manifest", {})
    columns = [c["name"] for c in manifest.get("schema", {}).get("columns", [])]
    
    # data_array is a simple list of lists: [["value1", "value2"], ["value1", "value2"]]
    rows = data.get("result", {}).get("data_array", [])
    
    all_facilities = [dict(zip(columns, r)) for r in rows]

    # 5. Filter by radius locally
    nearby = []
    for f in all_facilities:
        try:
            lat_str = f.get("lat")
            lon_str = f.get("lon")
            if not lat_str or not lon_str:
                continue
                
            f_lat = float(lat_str)
            f_lon = float(lon_str)
            
            dist = haversine(center_lat, center_lon, f_lat, f_lon)
            if dist <= radius_km:
                nearby.append({
                    "name"                : f.get("name"),
                    "facility_type"       : f.get("facilityTypeId"),
                    "city"                : f.get("address_city"),
                    "region"              : f.get("address_stateOrRegion"),
                    "lat"                 : f_lat,
                    "lon"                 : f_lon,
                    "distance_km"         : round(dist, 1),
                    "matched_capabilities": str(f.get("capability", ""))[:200]
                })
        except (ValueError, TypeError):
            continue

    # Sort by distance (closest first)
    nearby = sorted(nearby, key=lambda x: x["distance_km"])

    return json.dumps({
        "facilities": nearby,
        "count"     : len(nearby),
        "radius_km" : radius_km,
        "nearest"   : nearby[0] if nearby else None,
        "condition" : condition,
        "sql_used"  : sql_query
    })

except Exception as e:
    return json.dumps({"error": str(e), "count": 0})
$$
""")

print("find_facilities_near Function created")

# Tool 6: Find cold spots

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.find_cold_spots(
    procedure_or_capability STRING COMMENT 'The critical procedure or capability to check coverage for.
                                           Examples: "surgery", "emergency care", "dialysis",
                                           "maternity", "ICU", "blood transfusion", "caesarean".
                                           The tool checks which regions have NO facility
                                           offering this within the coverage radius.',

    coverage_radius_km      DOUBLE  COMMENT 'The radius in km that defines "accessible".
                                            WHO recommends 50km for emergency care.
                                            Use 50 for emergency services.
                                            Use 100 for specialist services.
                                            Use 25 for primary care.
                                            
                                            And, IF THE USER ASKS FOR TRAVEL TIME: 
                                            Assume an average regional travel speed of 50 km/h in Ghana. 
                                            For example: "1 hour" = 50.0 km. "2 hours" = 100.0 km. "30 mins" = 25.0 km.
                                            Translate the time into kilometers before calling this tool.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Identifies geographic cold spots — regions where a critical medical procedure
         or capability is absent or inaccessible within the coverage radius.
         Uses regional capitals as representative population center points.

         USE THIS TOOL WHEN:
         - "where is X procedure not available within Y km"
         - "find cold spots for [capability]"
         - "which regions lack access to [procedure]"
         - "geographic gaps in [service] coverage"
         - "medical deserts for [condition]"

         METHODOLOGY:
         1. Gets all facilities offering the procedure with their GPS coordinates
         2. For each of the 16 regional capitals, checks if any matching
            facility exists within coverage_radius_km
         3. Regions with NO matching facility within radius = cold spots

         RETURNS: JSON with
         - cold_spots: list of regions with no coverage
                       each has region, capital, population, 
                       nearest_facility (name + distance if exists)
         - covered_regions: list of regions that have coverage
         - coverage_pct: percentage of regions covered
         - total_population_in_cold_spots: sum of populations in uncovered regions
         - procedure: the procedure checked'
AS $$
import requests
import json
import math
import time

# WORKSPACE_URL = "your-WORKSPACE_URL"
# WAREHOUSE_ID = "your-WAREHOUSE_ID"
# DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

REGION_CAPITALS = {
    "Greater Accra Region": {"city": "Accra",        "lat": 5.6037,  "lon": -0.1870, "pop": 5455692},
    "Ashanti Region":       {"city": "Kumasi",       "lat": 6.6885,  "lon": -1.6244, "pop": 5440463},
    "Eastern Region":       {"city": "Koforidua",    "lat": 6.0940,  "lon": -0.2574, "pop": 2925653},
    "Central Region":       {"city": "Cape Coast",   "lat": 5.1054,  "lon": -1.2466, "pop": 2859821},
    "Northern Region":      {"city": "Tamale",       "lat": 9.4008,  "lon": -0.8393, "pop": 2310939},
    "Western Region":       {"city": "Sekondi",      "lat": 4.9341,  "lon": -1.7097, "pop": 2060585},
    "Volta Region":         {"city": "Ho",           "lat": 6.6011,  "lon":  0.4703, "pop": 1652772},
    "Upper East Region":    {"city": "Bolgatanga",   "lat": 10.7856, "lon": -0.8514, "pop": 1301226},
    "Bono Region":          {"city": "Sunyani",      "lat": 7.3349,  "lon": -2.3269, "pop": 1208649},
    "Bono East Region":     {"city": "Techiman",     "lat": 7.5833,  "lon": -1.9333, "pop": 1202739},
    "Upper West Region":    {"city": "Wa",           "lat": 10.0601, "lon": -2.5099, "pop": 901502},
    "Western North Region": {"city": "Sefwi Wiawso", "lat": 6.1667,  "lon": -2.4833, "pop": 880921},
    "Oti Region":           {"city": "Dambai",       "lat": 8.0667,  "lon":  0.1833, "pop": 747248},
    "Savannah Region":      {"city": "Damongo",      "lat": 9.0833,  "lon": -1.8167, "pop": 653266},
    "North East Region":    {"city": "Nalerigu",     "lat": 10.5306, "lon": -0.3686, "pop": 650856},
    "Ahafo Region":         {"city": "Goaso",        "lat": 6.8000,  "lon": -2.5167, "pop": 564662},
}

def haversine(lat1, lon1, lat2, lon2):
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat/2)**2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon/2)**2)
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

try:
    # 1. Build SQL Statement (WITH THE ARRAY FIX)
    proc_lower = procedure_or_capability.lower()
    sql_query = (
        f"SELECT name, address_stateOrRegion, lat, lon "
        f"FROM facilities3 "
        f"WHERE lat IS NOT NULL AND lon IS NOT NULL "
        f"AND lat != 0 AND lon != 0 "
        f"AND ("
        f"  exists(capability, x -> x ilike '%{proc_lower}%') OR "
        f"  exists(procedure, x -> x ilike '%{proc_lower}%') OR "
        f"  exists(specialties, x -> x ilike '%{proc_lower}%')"
        f")"
    )

    # 2. Call SQL Execution API
    payload = {
        "warehouse_id": WAREHOUSE_ID,
        "statement": sql_query,
        "wait_timeout": "30s"
    }
    
    resp = requests.post(f"https://{WORKSPACE_URL}/api/2.0/sql/statements", 
                         headers=headers, json=payload, timeout=40)
    
    if resp.status_code != 200:
        return json.dumps({"error": f"SQL API Error: {resp.text}"})

    data = resp.json()
    
    # Check status
    status = data.get("status", {}).get("state")
    if status != "SUCCEEDED":
        statement_id = data.get("statement_id")
        for _ in range(10):
            time.sleep(1)
            poll = requests.get(f"https://{WORKSPACE_URL}/api/2.0/sql/statements/{statement_id}", 
                                headers=headers)
            data = poll.json()
            if data.get("status", {}).get("state") == "SUCCEEDED":
                break

    # 3. Parse Results
    manifest = data.get("manifest", {})
    columns = [c["name"] for c in manifest.get("schema", {}).get("columns", [])]
    rows = data.get("result", {}).get("data_array", [])
    
    matching_facilities = [dict(zip(columns, r)) for r in rows]

    # 4. Geospatial Analysis
    cold_spots = []
    covered_regions = []
    cold_spot_pop = 0

    for region, capital in REGION_CAPITALS.items():
        cap_lat = capital["lat"]
        cap_lon = capital["lon"]
        
        nearest = None
        nearest_dist = float("inf")

        for f in matching_facilities:
            try:
                f_lat, f_lon = float(f["lat"]), float(f["lon"])
                dist = haversine(cap_lat, cap_lon, f_lat, f_lon)
                if dist < nearest_dist:
                    nearest_dist = dist
                    nearest = f
            except: continue

        if nearest_dist > coverage_radius_km:
            cold_spots.append({
                "region": region,
                "capital": capital["city"],
                "population": capital["pop"],
                "nearest_facility": nearest.get("name") if nearest else "None in dataset",
                "nearest_dist_km": round(nearest_dist, 1) if nearest else None
            })
            cold_spot_pop += capital["pop"]
        else:
            covered_regions.append(region)

    # 5. Final Output
    return json.dumps({
        "procedure": procedure_or_capability,
        "radius_km": coverage_radius_km,
        "cold_spots": sorted(cold_spots, key=lambda x: x["population"], reverse=True),
        "num_cold_spots": len(cold_spots),
        "coverage_pct": round(len(covered_regions) / 16 * 100, 1),
        "total_population_affected": cold_spot_pop
    })

except Exception as e:
    return json.dumps({"error": str(e)})
$$
""")

print("find_cold_spots UC Function created")

# Tool 7: Anomaly Analyzer

In [0]:
spark.sql(r'''
CREATE OR REPLACE FUNCTION workspace.default.analyze_anomalies(
    analysis_type STRING COMMENT 'Type of analysis to run. Must be one of:
                                  "unrealistic_claims"      — facilities claiming many specialties/procedures with no supporting capability or equipment
                                  "infrastructure_mismatch" — high-stakes claims (surgery/imaging/ICU) with no equipment to support them
                                  "correlation"             — how facility characteristics move together across facility types
                                  "procedure_concentration" — which high-stakes capabilities are concentrated in very few facilities
                                  "all"                     — run all four analyses and return combined results',

    location      STRING COMMENT 'Optional. Any location string to filter by.
                                  Matched as substring against osm_display_name and resolved_location_query.
                                  Works at any geographic level — neighbourhood, city, district, region.
                                  Pass NULL for all of Ghana.
                                  Examples:
                                  "Dansoman"             — neighbourhood
                                  "Accra"                — city
                                  "Ashanti"              — partial region name
                                  "Greater Accra Region" — full region
                                  "Ablekuma"             — district',

    min_score     INT COMMENT 'Optional. Minimum anomaly_score threshold. Default 30.
                                  Use 30 for broad sweep.
                                  Use 60 for only the most suspicious facilities.
                                  Use 0 to include all facilities regardless of score.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Detects anomalies and correlations in Ghana facility data using pre-computed columns.

         PRE-COMPUTED COLUMNS AVAILABLE:
         specialty_count, capability_count, procedure_count, equipment_count
         has_surgical_claim, has_imaging_claim, has_icu_claim, has_emergency_claim, has_maternity_claim
         unsupported_specialty_count     — specialties with no matching capability text
         capability_without_specialty_count — capability claims with no matching specialty
         proc_equip_ratio                — procedure count divided by equipment count
         claim_inflation_score           — 0-100 score for inflated claims vs infrastructure
         infrastructure_mismatch_score   — 0-100 score for high-stakes claims vs equipment
         anomaly_score                   — 0-100 combined score (primary signal)
         data_completeness_score         — 0-6 how complete the facility record is

         LOCATION FILTER works as substring match against osm_display_name and
         resolved_location_query — same approach as vector_search tool.
         More reliable than address columns which are often incomplete.

         USE THIS TOOL WHEN:
         - "which facilities have unrealistic claims"
         - "facilities where things do not add up"
         - "surgery claims but no equipment"
         - "imaging claims but no equipment listed"
         - "suspicious facilities in Accra / Kumasi / Northern Ghana"
         - "what correlates with what across facility types"
         - "which capabilities are concentrated in very few facilities"
         - "data quality issues"

         NOTE ON DATA LIMITATIONS:
         - numberDoctors is unknown for 99.7% of facilities — scores do not rely on it
         - capacity is unknown for 97.4% — scores do not rely on it
         - Primary signals are: specialty vs capability alignment, imaging/surgery claims vs equipment
         - data_completeness_score=0 means record is too sparse to trust anomaly_score

         RETURNS: JSON with analyses results, summary stats, methodology'
AS $$
import json
import requests
import re
import time

WORKSPACE_URL = ""
WAREHOUSE_ID="" 
DATABRICKS_TOKEN=""

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

# ── SQL Warehouse runner ──────────────────────────────────────
def run_sql(sql):
    BASE = f"https://{WORKSPACE_URL}/api/2.0/sql/statements"
    resp = requests.post(
        BASE,
        headers = headers,
        json    = {
            "warehouse_id": WAREHOUSE_ID,
            "statement"   : sql,
            "wait_timeout": "30s",
            "format"      : "JSON_ARRAY"
        },
        timeout = 40
    )
    data   = resp.json()
    status = data.get("status", {}).get("state")
    sid    = data.get("statement_id")

    while status in ("PENDING", "RUNNING"):
        time.sleep(2)
        poll   = requests.get(f"{BASE}/{sid}", headers=headers, timeout=15)
        data   = poll.json()
        status = data.get("status", {}).get("state")

    if status != "SUCCEEDED":
        err = data.get("status", {}).get("error", {}).get("message", "")
        raise Exception(f"SQL failed ({status}): {err}")

    columns = [
        c["name"] for c in data.get("manifest", {})
                                .get("schema", {})
                                .get("columns", [])
    ]
    rows = data.get("result", {}).get("data_array", []) or []
    return [dict(zip(columns, row)) for row in rows]

# ── Helper: build human-readable flags ───────────────────────
def build_flags(row):
    flags = []
    sc    = int(row.get("specialty_count")  or 0)
    cc    = int(row.get("capability_count") or 0)
    pc    = int(row.get("procedure_count")  or 0)
    ec    = int(row.get("equipment_count")  or 0)
    us    = int(row.get("unsupported_specialty_count") or 0)
    cws   = int(row.get("capability_without_specialty_count") or 0)
    hs    = int(row.get("has_surgical_claim") or 0)
    hi    = int(row.get("has_imaging_claim")  or 0)
    hu    = int(row.get("has_icu_claim")      or 0)
    he    = int(row.get("has_emergency_claim") or 0)
    ratio = float(row.get("proc_equip_ratio") or 0)

    if us > 0:
        flags.append(
            f"{us} claimed specialt{'y' if us == 1 else 'ies'} "
            f"have no supporting capability or procedure text"
        )
    if cws > 0:
        flags.append(
            f"{cws} high-stakes capability claim{'s' if cws > 1 else ''} "
            f"have no matching specialty declared"
        )
    if hs and ec == 0:
        flags.append("Claims surgical capability but lists no equipment")
    if hi and ec == 0:
        flags.append("Claims imaging (MRI/CT/X-ray) but lists no equipment")
    if hu and cc == 0:
        flags.append("Claims ICU but capability list is empty after cleaning")
    if he and cc == 0:
        flags.append("Claims emergency care but capability list is empty")
    if sc >= 3 and cc == 0:
        flags.append(f"Claims {sc} specialties but has no capability detail")
    if ratio > 5:
        flags.append(
            f"Procedure-to-equipment ratio is {ratio:.1f}x "
            f"({pc} procedures, only {ec} equipment items)"
        )
    if pc > 5 and ec == 0:
        flags.append(f"Claims {pc} procedures but lists no equipment")
    return flags

# ══════════════════════════════════════════════════════════════
# Step 1 — Resolve location filter to a set of pk_unique_ids
# Uses substring match on osm_display_name + resolved_location_query
# Same logic as vector_search tool
# ══════════════════════════════════════════════════════════════
id_filter_clause  = ""
location_matched  = None
location_count    = None

if location:
    # Normalize: lowercase, collapse spaces, hyphen → space
    loc_normalized = re.sub(r'[-_]', ' ', location.lower().strip())
    loc_normalized = re.sub(r'\s+', ' ', loc_normalized).strip()

    # Escape single quotes for SQL
    loc_sql = loc_normalized.replace("'", "''")

    # Get matching IDs via substring search on location columns
    matching_ids_rows = run_sql(f"""
        SELECT pk_unique_id
        FROM workspace.default.facilities3
        WHERE LOWER(osm_display_name)        LIKE '%{loc_sql}%'
           OR LOWER(resolved_location_query) LIKE '%{loc_sql}%'
           OR LOWER(address_city)            LIKE '%{loc_sql}%'
           OR LOWER(address_stateOrRegion)   LIKE '%{loc_sql}%'
    """)

    matching_ids  = [r["pk_unique_id"] for r in matching_ids_rows if r.get("pk_unique_id")]
    location_count = len(matching_ids)

    if not matching_ids:
        # Location not found — return early with helpful message
        return json.dumps({
            "error"            : f"No facilities found matching location '{location}'. "
                                 f"Try a broader term e.g. 'Accra' instead of a specific street.",
            "location_searched": location,
            "facilities_found" : 0
        })

    # Build IN clause — chunk if very large (SQL has limits)
    ids_str       = ", ".join([f"'{i}'" for i in matching_ids])
    id_filter_clause = f"AND pk_unique_id IN ({ids_str})"

# ── Build base WHERE with ID filter ──────────────────────────
threshold  = int(min_score or 30)
base_where = f"WHERE 1=1 {id_filter_clause}"

try:
    result = {}

    # ══════════════════════════════════════════════════════════
    # Analysis 1: Unrealistic claims
    # ══════════════════════════════════════════════════════════
    if analysis_type in ("unrealistic_claims", "all"):
        rows = run_sql(f"""
            SELECT
                name,
                facilityTypeId                         AS facility_type,
                address_city                           AS city,
                address_stateOrRegion                  AS region,
                osm_display_name,
                specialty_count,
                capability_count,
                procedure_count,
                equipment_count,
                unsupported_specialty_count,
                capability_without_specialty_count,
                proc_equip_ratio,
                claim_inflation_score                  AS score,
                data_completeness_score                AS completeness
            FROM workspace.default.facilities3
            {base_where}
            AND claim_inflation_score >= {threshold}
            ORDER BY claim_inflation_score DESC
            LIMIT 25
        """)

        for r in rows:
            r["flags"] = build_flags(r)

        reliable = [r for r in rows if int(r.get("completeness") or 0) >= 2]
        sparse   = [r for r in rows if int(r.get("completeness") or 0) < 2]

        result["unrealistic_claims"] = {
            "anomalies"              : reliable,
            "sparse_data_facilities" : sparse,
            "count"                  : len(reliable),
            "note"                   : (
                f"{len(sparse)} additional facilities flagged but excluded "
                f"due to low data completeness (score < 2) — likely missing "
                f"data rather than genuine anomalies"
            )
        }

    # ══════════════════════════════════════════════════════════
    # Analysis 2: Infrastructure mismatch
    # ══════════════════════════════════════════════════════════
    if analysis_type in ("infrastructure_mismatch", "all"):
        rows = run_sql(f"""
            SELECT
                name,
                facilityTypeId                    AS facility_type,
                address_city                      AS city,
                address_stateOrRegion             AS region,
                osm_display_name,
                has_surgical_claim,
                has_imaging_claim,
                has_icu_claim,
                has_emergency_claim,
                equipment_count,
                capability_count,
                specialty_count,
                infrastructure_mismatch_score     AS score,
                data_completeness_score           AS completeness
            FROM workspace.default.facilities3
            {base_where}
            AND infrastructure_mismatch_score >= {threshold}
            ORDER BY infrastructure_mismatch_score DESC
            LIMIT 25
        """)

        for r in rows:
            r["flags"] = build_flags(r)

        reliable = [r for r in rows if int(r.get("completeness") or 0) >= 2]
        sparse   = [r for r in rows if int(r.get("completeness") or 0) < 2]

        result["infrastructure_mismatch"] = {
            "anomalies"              : reliable,
            "sparse_data_facilities" : sparse,
            "count"                  : len(reliable),
            "note"                   : (
                f"{len(sparse)} excluded due to low completeness score"
            )
        }

    # ══════════════════════════════════════════════════════════
    # Analysis 3: Correlation
    # ══════════════════════════════════════════════════════════
    if analysis_type in ("correlation", "all"):
        corr_where = base_where.replace(
            "WHERE 1=1",
            "WHERE facilityTypeId IS NOT NULL"
        )
        rows = run_sql(f"""
            SELECT
                facilityTypeId                              AS facility_type,
                COUNT(*)                                    AS total_facilities,
                ROUND(AVG(specialty_count),  1)             AS avg_specialties,
                ROUND(AVG(capability_count), 1)             AS avg_capability_items,
                ROUND(AVG(procedure_count),  1)             AS avg_procedures,
                ROUND(AVG(equipment_count),  1)             AS avg_equipment,
                SUM(has_surgical_claim)                     AS count_surgical_claims,
                SUM(has_imaging_claim)                      AS count_imaging_claims,
                SUM(has_icu_claim)                          AS count_icu_claims,
                SUM(has_emergency_claim)                    AS count_emergency_claims,
                ROUND(AVG(anomaly_score),    1)             AS avg_anomaly_score,
                ROUND(AVG(data_completeness_score), 1)      AS avg_completeness,
                SUM(CASE WHEN anomaly_score >= 60
                         THEN 1 ELSE 0 END)                 AS high_suspicion_count
            FROM workspace.default.facilities3
            {corr_where}
            GROUP BY facilityTypeId
            ORDER BY avg_anomaly_score DESC
        """)

        result["correlation"] = {
            "by_facility_type": rows,
            "interpretation"  : (
                "Compare avg_specialties vs avg_capability_items vs avg_equipment "
                "to see claim-to-infrastructure gaps by facility type. "
                "High avg_anomaly_score for a type means systematic inflation. "
                "count_imaging_claims vs avg_equipment reveals which types "
                "claim imaging without hardware to support it."
            )
        }

    # ══════════════════════════════════════════════════════════
    # Analysis 4: Procedure concentration
    # ══════════════════════════════════════════════════════════
    if analysis_type in ("procedure_concentration", "all"):
        total_row = run_sql(f"""
            SELECT COUNT(*) AS total
            FROM workspace.default.facilities3
            {base_where}
        """)
        total = int(total_row[0]["total"]) if total_row else 797

        conc_rows = run_sql(f"""
            SELECT 'Surgical capability' AS claim_type,
                SUM(has_surgical_claim) AS facilities_claiming,
                ROUND(SUM(has_surgical_claim) * 100.0 / {total}, 1) AS pct_of_total,
                SUM(CASE WHEN has_surgical_claim = 1
                         AND equipment_count = 0 THEN 1 ELSE 0 END) AS with_no_equipment
            FROM workspace.default.facilities3 {base_where}

            UNION ALL
            SELECT 'Imaging (MRI/CT/X-ray)',
                SUM(has_imaging_claim),
                ROUND(SUM(has_imaging_claim) * 100.0 / {total}, 1),
                SUM(CASE WHEN has_imaging_claim = 1
                         AND equipment_count = 0 THEN 1 ELSE 0 END)
            FROM workspace.default.facilities3 {base_where}

            UNION ALL
            SELECT 'ICU / Intensive care',
                SUM(has_icu_claim),
                ROUND(SUM(has_icu_claim) * 100.0 / {total}, 1),
                SUM(CASE WHEN has_icu_claim = 1
                         AND capability_count = 0 THEN 1 ELSE 0 END)
            FROM workspace.default.facilities3 {base_where}

            UNION ALL
            SELECT 'Emergency care',
                SUM(has_emergency_claim),
                ROUND(SUM(has_emergency_claim) * 100.0 / {total}, 1),
                SUM(CASE WHEN has_emergency_claim = 1
                         AND capability_count = 0 THEN 1 ELSE 0 END)
            FROM workspace.default.facilities3 {base_where}

            UNION ALL
            SELECT 'Maternity / Obstetrics',
                SUM(has_maternity_claim),
                ROUND(SUM(has_maternity_claim) * 100.0 / {total}, 1),
                SUM(CASE WHEN has_maternity_claim = 1
                         AND capability_count = 0 THEN 1 ELSE 0 END)
            FROM workspace.default.facilities3 {base_where}

            ORDER BY facilities_claiming ASC
        """)

        result["procedure_concentration"] = {
            "distribution"    : conc_rows,
            "total_facilities": total,
            "interpretation"  : (
                "facilities_claiming = how many claim this capability. "
                "with_no_equipment = how many claim it but list zero equipment. "
                "Low facilities_claiming + high with_no_equipment = "
                "high-risk concentration: few facilities claim it and "
                "many of those claims may be unverifiable."
            )
        }

    # ══════════════════════════════════════════════════════════════
    # Analysis 5: Pattern mismatch — individual facilities
    # Returns specific facilities where correlated features don't match
    # This is what "things that shouldn't move together" actually needs
    # ══════════════════════════════════════════════════════════════
    if analysis_type in ("pattern_mismatch", "all"):
        rows = run_sql(f"""
            SELECT
                name,
                facilityTypeId                         AS facility_type,
                address_city                           AS city,
                address_stateOrRegion                  AS region,
                osm_display_name,
                specialty_count,
                capability_count,
                procedure_count,
                equipment_count,
                has_surgical_claim,
                has_imaging_claim,
                has_icu_claim,
                has_emergency_claim,
                has_maternity_claim,
                unsupported_specialty_count,
                capability_without_specialty_count,
                proc_equip_ratio,
                anomaly_score,
                claim_inflation_score,
                infrastructure_mismatch_score,
                data_completeness_score                AS completeness
            FROM workspace.default.facilities3
            {base_where}
            AND anomaly_score >= {threshold}
            AND data_completeness_score >= 2
            ORDER BY anomaly_score DESC
            LIMIT 30
        """)

        # Classify WHICH specific pattern each facility shows
        def classify_patterns(row):
            patterns = []
            sc    = int(row.get("specialty_count")  or 0)
            cc    = int(row.get("capability_count") or 0)
            pc    = int(row.get("procedure_count")  or 0)
            ec    = int(row.get("equipment_count")  or 0)
            us    = int(row.get("unsupported_specialty_count") or 0)
            cws   = int(row.get("capability_without_specialty_count") or 0)
            hs    = int(row.get("has_surgical_claim") or 0)
            hi    = int(row.get("has_imaging_claim")  or 0)
            hu    = int(row.get("has_icu_claim")      or 0)
            he    = int(row.get("has_emergency_claim") or 0)
            ratio = float(row.get("proc_equip_ratio") or 0)
            score = int(row.get("anomaly_score") or 0)

            # Pattern A: High specialty breadth + no capability support
            # "Claims to be a specialist center but has no clinical detail"
            if sc >= 3 and cc == 0:
                patterns.append({
                    "pattern"    : "Specialty breadth without capability detail",
                    "description": f"Claims {sc} specialties but has no capability records — "
                                f"cannot verify these claims against clinical evidence",
                    "severity"   : "high" if sc >= 5 else "medium"
                })

            # Pattern B: Imaging claims + no equipment
            # "Strongest hardware signal — MRI/CT needs physical machines"
            if hi and ec == 0:
                patterns.append({
                    "pattern"    : "Imaging claim without equipment",
                    "description": f"Claims MRI/CT/X-ray/ultrasound capability but lists "
                                f"zero equipment — imaging is impossible without hardware",
                    "severity"   : "high"
                })

            # Pattern C: Surgical claims + no equipment
            if hs and ec == 0:
                patterns.append({
                    "pattern"    : "Surgical claim without equipment",
                    "description": f"Claims surgical capability but lists no equipment — "
                                f"surgery requires instruments, sterilization, anesthesia equipment",
                    "severity"   : "high"
                })

            # Pattern D: ICU claim + empty capability
            if hu and cc == 0:
                patterns.append({
                    "pattern"    : "ICU claim without supporting capability detail",
                    "description": f"Claims ICU/intensive care but capability list is empty — "
                                f"cannot verify what ICU services are actually available",
                    "severity"   : "high"
                })

            # Pattern E: Specialty-capability misalignment
            # "Claims cardiology specialty but capability says nothing about hearts"
            if us >= 2:
                patterns.append({
                    "pattern"    : "Specialty-capability misalignment",
                    "description": f"{us} claimed specialt{'y' if us == 1 else 'ies'} have "
                                f"no supporting text in capability or procedure records — "
                                f"specialties appear disconnected from clinical evidence",
                    "severity"   : "high" if us >= 3 else "medium"
                })

            # Pattern F: Reverse misalignment
            # "Capability mentions cardiac care but no cardiology specialty listed"
            if cws >= 2:
                patterns.append({
                    "pattern"    : "Capability without declared specialty",
                    "description": f"{cws} high-stakes capability claim{'s' if cws > 1 else ''} "
                                f"have no corresponding specialty declared — "
                                f"e.g. cardiac capability listed but no cardiology specialty",
                    "severity"   : "medium"
                })

            # Pattern G: High procedure-to-equipment ratio
            # "Claims many procedures but barely any equipment to do them with"
            if ratio > 5 and ec > 0:
                patterns.append({
                    "pattern"    : "Disproportionate procedure-to-equipment ratio",
                    "description": f"Claims {pc} procedures with only {ec} equipment items "
                                f"({ratio:.1f}x ratio) — most procedures require specific equipment",
                    "severity"   : "medium"
                })

            # Pattern H: Many procedures, zero equipment
            if pc > 5 and ec == 0:
                patterns.append({
                    "pattern"    : "Procedures without any equipment",
                    "description": f"Claims {pc} specific procedures but lists no equipment — "
                                f"clinical procedures require physical instruments and devices",
                    "severity"   : "medium"
                })

            return patterns

        # Apply pattern classification to each facility
        enriched_rows = []
        for r in rows:
            patterns = classify_patterns(r)
            if patterns:  # only include if we found a specific pattern
                enriched_rows.append({
                    "name"                : r.get("name"),
                    "facility_type"       : r.get("facility_type"),
                    "city"                : r.get("city"),
                    "region"              : r.get("region"),
                    "anomaly_score"       : r.get("anomaly_score"),
                    "completeness"        : r.get("completeness"),
                    "specialty_count"     : r.get("specialty_count"),
                    "capability_count"    : r.get("capability_count"),
                    "procedure_count"     : r.get("procedure_count"),
                    "equipment_count"     : r.get("equipment_count"),
                    "patterns_found"      : patterns,
                    "primary_pattern"     : patterns[0]["pattern"] if patterns else None,
                    "pattern_count"       : len(patterns)
                })

        # Sort by number of patterns found then anomaly score
        enriched_rows.sort(
            key=lambda x: (x["pattern_count"], x["anomaly_score"]),
            reverse=True
        )

        result["pattern_mismatch"] = {
            "facilities"   : enriched_rows,
            "count"        : len(enriched_rows),
            "note"         : (
                "Each facility is listed with specific patterns where expected "
                "correlated features don't match. Only facilities with "
                f"data_completeness_score >= 2 and anomaly_score >= {threshold} "
                "are included to avoid flagging incomplete records as anomalies."
            )
        }

    # ══════════════════════════════════════════════════════════
    # Overall summary
    # ══════════════════════════════════════════════════════════
    summary = run_sql(f"""
        SELECT
            COUNT(*)                                                      AS total_facilities,
            SUM(CASE WHEN anomaly_score >= 60 THEN 1 ELSE 0 END)          AS high_suspicion,
            SUM(CASE WHEN anomaly_score BETWEEN 35 AND 59
                     THEN 1 ELSE 0 END)                                   AS medium_suspicion,
            SUM(CASE WHEN anomaly_score < 35 THEN 1 ELSE 0 END)           AS low_or_clean,
            SUM(CASE WHEN data_completeness_score <= 1
                     THEN 1 ELSE 0 END)                                   AS very_sparse_records,
            ROUND(AVG(anomaly_score), 1)                                   AS avg_anomaly_score,
            ROUND(AVG(data_completeness_score), 1)                         AS avg_completeness
        FROM workspace.default.facilities3
        {base_where}
    """)

    result["summary"]          = summary[0] if summary else {}
    result["location_filter"]  = {
        "searched"          : location,
        "facilities_matched": location_count,
        "method"            : "substring match on osm_display_name, resolved_location_query, address_city, address_stateOrRegion"
    } if location else {"searched": None, "facilities_matched": "all 797"}

    result["methodology"] = {
        "primary_signals"  : [
            "unsupported_specialty_count: specialties with no matching capability text",
            "capability_without_specialty_count: capability claims with no matching specialty",
            "has_imaging_claim + equipment_count=0: imaging claimed without equipment"
        ],
        "secondary_signals": [
            "proc_equip_ratio: procedure count vs equipment count",
            "high specialty_count with empty capability_count"
        ],
        "not_used"         : [
            "numberDoctors: 99.7% unknown in this dataset",
            "capacity: 97.4% unknown in this dataset"
        ],
        "completeness_note": (
            "data_completeness_score < 2 means the record has too little data "
            "to distinguish a real anomaly from missing information. "
            "Always interpret anomaly_score alongside completeness_score."
        )
    }

    return json.dumps(result)

except Exception as e:
    return json.dumps({"error": str(e), "analysis_type": analysis_type})
$$
''')

print("analyze_anomalies UC Function created")